# Transformer — Evaluation on Held-Out Eval Set
Evaluates the best Transformer checkpoint (trained on 100% of training data) on the full ASVspoof2019 LA eval set.
All figures are saved individually to `./experiments/evaluation_transformer/figures/`

In [1]:
import os
import math
import torch
import torch.nn as nn
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')  # non-interactive backend — ensures saving works reliably
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import torchaudio.transforms as T
from torch.utils.data import DataLoader, Dataset
from pathlib import Path

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

Device: cuda
GPU: NVIDIA GeForce RTX 3050 Laptop GPU


In [2]:
# ── CONFIG ──────────────────────────────────────────────────────────────────
DATA_ROOT         = r'G:\.shortcut-targets-by-id\1ua-L70AEx_VqEEKdgxjSMzr-wEX6GnxK\voice-manipulation-detection'
CACHE_DIR         = './lfcc_cache'
CHECKPOINT_PATH   = './models/checkpoints/transformer/transformer_best_model.pt'
FIG_DIR           = Path('./experiments/evaluation_transformer/figures')
FIG_DIR.mkdir(parents=True, exist_ok=True)

SUBSET_FRACTION   = 1.0
MAX_LENGTH        = 64000
BATCH_SIZE        = 32

print(f'Checkpoint : {CHECKPOINT_PATH}')
print(f'Checkpoint exists: {os.path.exists(CHECKPOINT_PATH)}')
print(f'Figures will be saved to: {FIG_DIR}')

Checkpoint : ./models/checkpoints/transformer/transformer_best_model.pt
Checkpoint exists: True
Figures will be saved to: experiments\evaluation_transformer\figures


In [3]:
# ── DATASET ─────────────────────────────────────────────────────────────────
class ASVspoofDataset(Dataset):
    def __init__(self, data_root, split='eval', cache_dir='./lfcc_cache',
                 subset_fraction=1.0, max_length=64000):
        import soundfile as sf
        import librosa
        self.sf       = sf
        self.librosa  = librosa
        self.root_dir = data_root
        self.cache_dir = Path(cache_dir) / split
        self.cache_dir.mkdir(parents=True, exist_ok=True)

        protocol_dir   = os.path.join(data_root, 'data', 'raw', 'ASVspoof2019',
                                      'LA', 'LA', 'ASVspoof2019_LA_cm_protocols')
        protocol_files = {
            'train': 'ASVspoof2019.LA.cm.train.trn.txt',
            'dev':   'ASVspoof2019.LA.cm.dev.trl.txt',
            'eval':  'ASVspoof2019.LA.cm.eval.trl.txt'
        }
        protocol_path = os.path.join(protocol_dir, protocol_files[split])
        if not os.path.exists(protocol_path):
            raise FileNotFoundError(f'Protocol file not found: {protocol_path}')

        self.metadata = pd.read_csv(
            protocol_path, sep=' ', header=None,
            names=['speaker', 'filename', 'system', 'null', 'label']
        )
        if subset_fraction < 1.0:
            self.metadata = self.metadata.sample(
                frac=subset_fraction, random_state=42).reset_index(drop=True)

        self.audio_dir  = os.path.join(data_root, 'data', 'raw', 'ASVspoof2019',
                                       'LA', 'LA', f'ASVspoof2019_LA_{split}', 'flac')
        self.max_length = max_length

    def __len__(self):
        return len(self.metadata)

    def __getitem__(self, idx):
        row        = self.metadata.iloc[idx]
        cache_path = self.cache_dir / f"{row['filename']}.pt"

        if cache_path.exists():
            try:
                data = torch.load(cache_path, weights_only=True)
            except TypeError:
                data = torch.load(cache_path)
            return {
                'lfcc':     data['lfcc'],
                'label':    torch.tensor(data['label'], dtype=torch.long)
                            if not isinstance(data['label'], torch.Tensor)
                            else data['label'],
                'is_lfcc':  True,
                'filename': row['filename'],
                'system':   row['system']
            }

        audio_path = os.path.join(self.audio_dir, row['filename'] + '.flac')
        try:
            audio_data, sr = self.sf.read(audio_path)
            if len(audio_data.shape) > 1:
                audio_data = audio_data.mean(axis=1)
            if sr != 16000:
                audio_data = self.librosa.resample(
                    audio_data, orig_sr=sr, target_sr=16000)
            waveform = torch.from_numpy(audio_data).float().unsqueeze(0)
            if waveform.shape[1] > self.max_length:
                waveform = waveform[:, :self.max_length]
            else:
                pad = self.max_length - waveform.shape[1]
                waveform = torch.nn.functional.pad(waveform, (0, pad))
        except Exception:
            waveform = torch.zeros(1, self.max_length)

        label = 0 if row['label'] == 'bonafide' else 1
        return {
            'waveform': waveform,
            'label':    torch.tensor(label, dtype=torch.long),
            'is_lfcc':  False,
            'filename': row['filename'],
            'system':   row['system']
        }

In [4]:
# ── LFCC EXTRACTOR + COLLATE ─────────────────────────────────────────────────
class LFCCExtractor(nn.Module):
    def __init__(self, n_lfcc=60, n_fft=512, hop_length=160):
        super().__init__()
        self.n_lfcc = n_lfcc
        self.spec   = T.Spectrogram(n_fft=n_fft, hop_length=hop_length, power=2.0)
        self.register_buffer('dct_mat', None)

    def _create_dct_matrix(self, n_freqs, n_lfcc):
        n   = torch.arange(float(n_freqs)).unsqueeze(1)
        k   = torch.arange(float(n_lfcc)).unsqueeze(0)
        dct = torch.cos(torch.pi / float(n_freqs) * (n + 0.5) * k)
        return dct / torch.sqrt(torch.sum(dct**2, dim=0, keepdim=True))

    def forward(self, waveform):
        spec = self.spec(waveform).squeeze(1)
        spec = torch.log(torch.sqrt(spec) + 1e-10)
        if self.dct_mat is None:
            self.dct_mat = self._create_dct_matrix(
                spec.shape[1], self.n_lfcc).to(spec.device)
        spec = spec.transpose(1, 2)
        return torch.matmul(spec, self.dct_mat).transpose(1, 2)


def make_collate(lfcc_extractor, device):
    def collate_fn(batch):
        labels    = torch.stack([b['label']    for b in batch]).to(device)
        filenames = [b['filename'] for b in batch]
        systems   = [b['system']   for b in batch]
        if batch[0].get('is_lfcc', False):
            features = torch.stack([b['lfcc'] for b in batch]).to(device)
            features = features.transpose(1, 2)   # [B,60,T] -> [B,T,60]
            return features, labels, filenames, systems
        waveforms = torch.stack([b['waveform'] for b in batch]).to(device)
        with torch.no_grad():
            features = lfcc_extractor(waveforms).transpose(1, 2)
        return features, labels, filenames, systems
    return collate_fn


lfcc_extractor = LFCCExtractor(n_lfcc=60).to(device)
print('LFCC extractor ready')

LFCC extractor ready


In [5]:
# ── TRANSFORMER ARCHITECTURE ─────────────────────────────────────────────────
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=420, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)
        pe       = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() *
                             (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1), :])


class TransformerSpoofDetector(nn.Module):
    def __init__(self, input_dim=60, d_model=128, nhead=4, num_layers=2,
                 dim_feedforward=256, dropout=0.3, num_classes=2):
        super().__init__()
        self.input_projection = nn.Sequential(
            nn.Linear(input_dim, d_model),
            nn.LayerNorm(d_model),
            nn.Dropout(dropout)
        )
        self.pos_encoder = PositionalEncoding(d_model, max_len=420, dropout=dropout)
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead, dim_feedforward=dim_feedforward,
            dropout=dropout, activation='gelu', batch_first=True, norm_first=True
        )
        self.transformer_encoder = nn.TransformerEncoder(
            encoder_layer, num_layers=num_layers, norm=nn.LayerNorm(d_model)
        )
        self.cls_token  = nn.Parameter(torch.randn(1, 1, d_model))
        self.classifier = nn.Sequential(
            nn.Dropout(dropout),
            nn.Linear(d_model, d_model // 2),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(d_model // 2, num_classes)
        )

    def forward(self, x):
        x   = self.input_projection(x)
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x   = torch.cat([cls, x], dim=1)
        x   = self.pos_encoder(x)
        x   = self.transformer_encoder(x)
        return self.classifier(x[:, 0, :])

print('Transformer architecture loaded')

Transformer architecture loaded


In [6]:
# ── LOAD MODEL ───────────────────────────────────────────────────────────────
model = TransformerSpoofDetector(
    input_dim=60, d_model=128, nhead=4, num_layers=2,
    dim_feedforward=256, dropout=0.3, num_classes=2
)
ckpt = torch.load(CHECKPOINT_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt['model_state_dict'])
model = model.to(device)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'Loaded from epoch  : {ckpt.get("epoch", "?")}')
print(f'Dev EER (saved)    : {ckpt.get("eer", "?"):.4f}')
print(f'Dev Acc (saved)    : {ckpt.get("accuracy", "?"):.2f}%')
print(f'Total params       : {total_params:,}')

Loaded from epoch  : 13
Dev EER (saved)    : 0.0180
Dev Acc (saved)    : 97.40%
Total params       : 281,794


C:\Users\PC\miniconda3\envs\voice-manipulation-detection\Lib\site-packages\torch\nn\modules\transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


In [8]:
# ── LOAD EVAL DATASET + CACHE CHECK ─────────────────────────────────────────
print('Loading eval dataset...')
eval_dataset = ASVspoofDataset(
    DATA_ROOT, split='eval', cache_dir=CACHE_DIR,
    subset_fraction=SUBSET_FRACTION, max_length=MAX_LENGTH
)

eval_labels   = eval_dataset.metadata['label'].values
eval_bonafide = (eval_labels == 'bonafide').sum()
eval_spoof    = (eval_labels == 'spoof').sum()
print(f'Eval samples : {len(eval_dataset):,}')
print(f'  Bonafide   : {eval_bonafide:,}')
print(f'  Spoof      : {eval_spoof:,}')

# Verify cache completeness before running eval
missing = 0
for idx in range(len(eval_dataset)):
    row        = eval_dataset.metadata.iloc[idx]
    cache_path = eval_dataset.cache_dir / f"{row['filename']}.pt"
    if not cache_path.exists():
        missing += 1

if missing > 0:
    print(f'\nWARNING: {missing} cache files missing — running precompute...')
    for idx in range(len(eval_dataset)):
        row        = eval_dataset.metadata.iloc[idx]
        cache_path = eval_dataset.cache_dir / f"{row['filename']}.pt"
        if cache_path.exists():
            continue
        audio_path = os.path.join(eval_dataset.audio_dir, row['filename'] + '.flac')
        try:
            import soundfile as sf, librosa
            audio_data, sr = sf.read(audio_path)
            if len(audio_data.shape) > 1:
                audio_data = audio_data.mean(axis=1)
            if sr != 16000:
                audio_data = librosa.resample(audio_data, orig_sr=sr, target_sr=16000)
            waveform = torch.from_numpy(audio_data).float().unsqueeze(0)
            if waveform.shape[1] > MAX_LENGTH:
                waveform = waveform[:, :MAX_LENGTH]
            else:
                pad = MAX_LENGTH - waveform.shape[1]
                waveform = torch.nn.functional.pad(waveform, (0, pad))
        except Exception:
            waveform = torch.zeros(1, MAX_LENGTH)
        waveform = waveform.to(device).unsqueeze(0)
        with torch.no_grad():
            lfcc = lfcc_extractor(waveform).squeeze(0).cpu()
        label = 0 if row['label'] == 'bonafide' else 1
        torch.save({'lfcc': lfcc, 'label': label}, cache_path)
        if (idx + 1) % 1000 == 0:
            print(f'  Cached {idx+1}/{len(eval_dataset)}')
    print('Cache complete.')
else:
    print(f'Cache: all {len(eval_dataset):,} files present ')

Loading eval dataset...
Eval samples : 71,237
  Bonafide   : 7,355
  Spoof      : 63,882
Cache: all 71,237 files present 


In [9]:
# ── EER + METRICS ────────────────────────────────────────────────────────────
def calculate_eer(labels, scores):
    labels = labels.astype(int)
    scores = scores.astype(float)
    n_spoof    = np.sum(labels == 1)
    n_bonafide = np.sum(labels == 0)
    if n_spoof == 0 or n_bonafide == 0:
        return 0.5
    order          = np.argsort(-scores)
    labels_sorted  = labels[order]
    tps = np.cumsum(labels_sorted == 1)
    fps = np.cumsum(labels_sorted == 0)
    tpr = tps / n_spoof
    fpr = fps / n_bonafide
    fnr = 1 - tpr
    idx = np.nanargmin(np.abs(fnr - fpr))
    eer = (fnr[idx] + fpr[idx]) / 2
    return max(0.0, min(1.0, float(eer)))


def run_evaluation(model, dataset, device):
    collate_fn  = make_collate(lfcc_extractor, device)
    data_loader = DataLoader(
        dataset, batch_size=BATCH_SIZE, shuffle=False,
        num_workers=0, pin_memory=False, collate_fn=collate_fn
    )
    model.eval()
    all_scores, all_labels, all_preds, all_systems = [], [], [], []
    correct, total = 0, 0

    print(f'Running inference on {len(dataset):,} samples...')
    with torch.no_grad():
        for i, (data, target, filenames, systems) in enumerate(data_loader):
            output    = model(data)
            probs     = torch.softmax(output, dim=1)
            scores    = probs[:, 1].cpu().numpy()
            _, preds  = torch.max(output, 1)
            total    += target.size(0)
            correct  += (preds == target).sum().item()
            all_scores.extend(scores)
            all_labels.extend(target.cpu().numpy())
            all_preds.extend(preds.cpu().numpy())
            all_systems.extend(systems)
            if (i + 1) % 100 == 0:
                print(f'  Batch {i+1}/{len(data_loader)}  '
                      f'({(i+1)*BATCH_SIZE:,}/{len(dataset):,} samples)', flush=True)

    all_labels  = np.array(all_labels)
    all_scores  = np.array(all_scores)
    all_preds   = np.array(all_preds)
    all_systems = np.array(all_systems)

    accuracy  = 100 * correct / total
    eer       = calculate_eer(all_labels, all_scores)
    tp = np.sum((all_preds == 1) & (all_labels == 1))
    tn = np.sum((all_preds == 0) & (all_labels == 0))
    fp = np.sum((all_preds == 1) & (all_labels == 0))
    fn = np.sum((all_preds == 0) & (all_labels == 1))
    precision = tp / (tp + fp) if (tp + fp) > 0 else 0
    recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
    f1        = 2*precision*recall / (precision+recall) if (precision+recall) > 0 else 0

    return {
        'accuracy': accuracy, 'eer': eer,
        'precision': precision, 'recall': recall, 'f1': f1,
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'labels': all_labels, 'scores': all_scores,
        'preds': all_preds, 'systems': all_systems, 'total': total
    }

print('Evaluation functions ready')

Evaluation functions ready


In [10]:
# ── RUN EVALUATION ───────────────────────────────────────────────────────────
import time
t0      = time.time()
metrics = run_evaluation(model, eval_dataset, device)
elapsed = time.time() - t0

print(f'\n{"="*55}')
print(f'TRANSFORMER — EVAL SET RESULTS')
print(f'{"="*55}')
print(f'Samples    : {metrics["total"]:,}')
print(f'Accuracy   : {metrics["accuracy"]:.2f}%')
print(f'EER        : {metrics["eer"]:.4f}  ({metrics["eer"]*100:.2f}%)')
print(f'Precision  : {metrics["precision"]:.4f}')
print(f'Recall     : {metrics["recall"]:.4f}')
print(f'F1 Score   : {metrics["f1"]:.4f}')
print(f'Confusion  : TP={metrics["tp"]} TN={metrics["tn"]} FP={metrics["fp"]} FN={metrics["fn"]}')
print(f'Time       : {elapsed/60:.1f} min')

Running inference on 71,237 samples...
  Batch 100/2227  (3,200/71,237 samples)
  Batch 200/2227  (6,400/71,237 samples)
  Batch 300/2227  (9,600/71,237 samples)
  Batch 400/2227  (12,800/71,237 samples)
  Batch 500/2227  (16,000/71,237 samples)
  Batch 600/2227  (19,200/71,237 samples)
  Batch 700/2227  (22,400/71,237 samples)
  Batch 800/2227  (25,600/71,237 samples)
  Batch 900/2227  (28,800/71,237 samples)
  Batch 1000/2227  (32,000/71,237 samples)
  Batch 1100/2227  (35,200/71,237 samples)
  Batch 1200/2227  (38,400/71,237 samples)
  Batch 1300/2227  (41,600/71,237 samples)
  Batch 1400/2227  (44,800/71,237 samples)
  Batch 1500/2227  (48,000/71,237 samples)
  Batch 1600/2227  (51,200/71,237 samples)
  Batch 1700/2227  (54,400/71,237 samples)
  Batch 1800/2227  (57,600/71,237 samples)
  Batch 1900/2227  (60,800/71,237 samples)
  Batch 2000/2227  (64,000/71,237 samples)
  Batch 2100/2227  (67,200/71,237 samples)
  Batch 2200/2227  (70,400/71,237 samples)

TRANSFORMER — EVAL SET RES

In [13]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = np.array([[metrics['tn'], metrics['fp']],
               [metrics['fn'], metrics['tp']]])
im = ax.imshow(cm, interpolation='nearest', cmap='Blues')
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['Predicted\nBonafide', 'Predicted\nSpoof'], fontsize=11)
ax.set_yticklabels(['Actual\nBonafide', 'Actual\nSpoof'], fontsize=11)
ax.set_title(f'Transformer — Confusion Matrix\nAcc={metrics["accuracy"]:.2f}%  EER={metrics["eer"]:.4f}', fontsize=13)
for i in range(2):
    for j in range(2):
        color = 'white' if cm[i, j] > cm.max() / 2 else 'black'
        ax.text(j, i, f'{cm[i,j]:,}', ha='center', va='center',
                color=color, fontsize=14, fontweight='bold')
plt.tight_layout()
save_path = FIG_DIR / '01_confusion_matrix.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

Saved: experiments\evaluation_transformer\figures\01_confusion_matrix.png


C:\Users\PC\AppData\Local\Temp\ipykernel_4064\1372235154.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [14]:
# ── FIG 2: SCORE DISTRIBUTION ────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(8, 5))
bonafide_scores = metrics['scores'][metrics['labels'] == 0]
spoof_scores    = metrics['scores'][metrics['labels'] == 1]

ax.hist(bonafide_scores, bins=80, alpha=0.65, label=f'Bonafide (n={len(bonafide_scores):,})',
        color='#22c55e', density=True, edgecolor='none')
ax.hist(spoof_scores,    bins=80, alpha=0.65, label=f'Spoof    (n={len(spoof_scores):,})',
        color='#ef4444', density=True, edgecolor='none')
ax.axvline(x=0.5, color='black', linestyle='--', linewidth=1.5, label='Decision threshold (0.5)')
ax.set_xlabel('Spoof Probability Score', fontsize=12)
ax.set_ylabel('Density', fontsize=12)
ax.set_title('Transformer — Score Distribution (Eval Set)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
plt.tight_layout()
save_path = FIG_DIR / '02_score_distribution.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

Saved: experiments\evaluation_transformer\figures\02_score_distribution.png


C:\Users\PC\AppData\Local\Temp\ipykernel_4064\1459701648.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [15]:
# ── FIG 3: DET CURVE ─────────────────────────────────────────────────────────
labels  = metrics['labels'].astype(int)
scores  = metrics['scores'].astype(float)
n_spoof    = np.sum(labels == 1)
n_bonafide = np.sum(labels == 0)

thresholds = np.sort(scores)
step       = max(1, len(thresholds) // 500)
fpr_list, fnr_list = [], []
for t in thresholds[::step]:
    fp = np.sum((scores >= t) & (labels == 0))
    fn = np.sum((scores <  t) & (labels == 1))
    fpr_list.append(fp / n_bonafide)
    fnr_list.append(fn / n_spoof)

fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(fpr_list, fnr_list, color='#6c63ff', linewidth=2.5,
        label=f'Transformer  EER={metrics["eer"]:.4f}')
ax.plot([0, 1], [0, 1], 'k--', alpha=0.4, label='EER line')
ax.scatter([metrics['eer']], [metrics['eer']], color='red', s=80, zorder=5,
           label=f'EER point ({metrics["eer"]*100:.2f}%)')
ax.set_xlabel('False Positive Rate (FPR)', fontsize=12)
ax.set_ylabel('False Negative Rate (FNR)', fontsize=12)
ax.set_title('Transformer — DET Curve (Eval Set)', fontsize=13)
ax.legend(fontsize=11)
ax.grid(True, alpha=0.3)
ax.set_xlim([0, 0.6])
ax.set_ylim([0, 0.6])
plt.tight_layout()
save_path = FIG_DIR / '03_det_curve.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

Saved: experiments\evaluation_transformer\figures\03_det_curve.png


C:\Users\PC\AppData\Local\Temp\ipykernel_4064\1659286169.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [17]:
# ── FIG 4: PER-ATTACK-TYPE ANALYSIS ─────────────────────────────────────────
systems = metrics['systems']
labels  = metrics['labels']
preds   = metrics['preds']
scores  = metrics['scores']

unique_systems = sorted(set(systems))
print(f'Systems found: {unique_systems}')
print(f'{"System":<10} {"Count":>8} {"Acc":>8} {"EER":>8}')
print('-' * 38)

sys_names, sys_accs, sys_eers, sys_counts = [], [], [], []
for sys in unique_systems:
    mask       = systems == sys
    count      = mask.sum()
    if count == 0:
        continue
    sys_labels = labels[mask]
    sys_preds  = preds[mask]
    sys_scores = scores[mask]
    acc        = 100 * np.mean(sys_labels == sys_preds)
    eer        = calculate_eer(sys_labels, sys_scores) \
                 if len(set(sys_labels.tolist())) > 1 else float('nan')
    sys_names.append(sys)
    sys_accs.append(acc)
    sys_eers.append(eer)
    sys_counts.append(count)
    print(f'{sys:<10} {count:>8} {acc:>7.2f}% {eer:>8.4f}')

Systems found: [np.str_('-')]
System        Count      Acc      EER
--------------------------------------
-             71237   87.11%   0.2676


In [18]:
# ── FIG 4 PLOT: PER-ATTACK ACCURACY ─────────────────────────────────────────
valid = [(n, a, e, c) for n, a, e, c in
         zip(sys_names, sys_accs, sys_eers, sys_counts)
         if not np.isnan(e)]

if valid:
    v_names, v_accs, v_eers, v_counts = zip(*valid)
    x    = np.arange(len(v_names))
    colors_acc = ['#22c55e' if a >= 80 else '#fbbf24' if a >= 50 else '#ef4444'
                  for a in v_accs]

    fig, axes = plt.subplots(2, 1, figsize=(12, 9))

    # Accuracy per system
    bars = axes[0].bar(x, v_accs, color=colors_acc, edgecolor='black', linewidth=0.5)
    axes[0].axhline(y=50, color='black', linestyle='--', alpha=0.4, label='50% baseline')
    axes[0].set_xticks(x)
    axes[0].set_xticklabels(v_names, fontsize=10)
    axes[0].set_ylabel('Accuracy (%)', fontsize=11)
    axes[0].set_title('Transformer — Per-Attack-Type Accuracy (Eval Set)', fontsize=13)
    axes[0].set_ylim(0, 110)
    axes[0].legend(fontsize=10)
    axes[0].grid(True, alpha=0.3, axis='y')
    for bar, acc in zip(bars, v_accs):
        axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
                     f'{acc:.1f}%', ha='center', va='bottom', fontsize=9)

    # EER per system
    colors_eer = ['#22c55e' if e <= 0.1 else '#fbbf24' if e <= 0.3 else '#ef4444'
                  for e in v_eers]
    bars2 = axes[1].bar(x, [e*100 for e in v_eers], color=colors_eer,
                        edgecolor='black', linewidth=0.5)
    axes[1].set_xticks(x)
    axes[1].set_xticklabels(v_names, fontsize=10)
    axes[1].set_ylabel('EER (%)', fontsize=11)
    axes[1].set_title('Transformer — Per-Attack-Type EER (Eval Set)', fontsize=13)
    axes[1].grid(True, alpha=0.3, axis='y')
    for bar, eer in zip(bars2, v_eers):
        axes[1].text(bar.get_x() + bar.get_width()/2, eer*100 + 0.3,
                     f'{eer*100:.1f}%', ha='center', va='bottom', fontsize=9)

    green  = mpatches.Patch(color='#22c55e', label='Good  (Acc≥80% / EER≤10%)')
    yellow = mpatches.Patch(color='#fbbf24', label='OK    (Acc≥50% / EER≤30%)')
    red    = mpatches.Patch(color='#ef4444', label='Poor  (Acc<50%  / EER>30%)')
    axes[1].legend(handles=[green, yellow, red], fontsize=9, loc='upper right')

    plt.tight_layout()
    save_path = FIG_DIR / '04_per_attack_analysis.png'
    plt.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.show()
    print(f'Saved: {save_path}')
else:
    print('No valid per-attack data to plot — all systems may be bonafide ("-")')

Saved: experiments\evaluation_transformer\figures\04_per_attack_analysis.png


C:\Users\PC\AppData\Local\Temp\ipykernel_4064\53162942.py:50: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [19]:
# ── FIG 5: METRICS SUMMARY BAR ───────────────────────────────────────────────
metric_names = ['Accuracy', 'Precision', 'Recall', 'F1 Score']
metric_vals  = [
    metrics['accuracy'] / 100,
    metrics['precision'],
    metrics['recall'],
    metrics['f1']
]
colors_bar = ['#6c63ff', '#22c55e', '#f59e0b', '#3b82f6']

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(metric_names, metric_vals, color=colors_bar,
              edgecolor='black', linewidth=0.5)
ax.set_ylim(0, 1.15)
ax.set_ylabel('Score', fontsize=12)
ax.set_title(f'Transformer — Performance Summary (Eval Set)\nEER={metrics["eer"]*100:.2f}%',
             fontsize=13)
ax.grid(True, alpha=0.3, axis='y')
for bar, val in zip(bars, metric_vals):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.3f}', ha='center', va='bottom', fontsize=12, fontweight='bold')

# Add EER as annotation
ax.text(0.98, 0.97, f'EER = {metrics["eer"]*100:.2f}%',
        transform=ax.transAxes, ha='right', va='top',
        fontsize=12, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='#fef3c7', edgecolor='#f59e0b'))

plt.tight_layout()
save_path = FIG_DIR / '05_metrics_summary.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

Saved: experiments\evaluation_transformer\figures\05_metrics_summary.png


C:\Users\PC\AppData\Local\Temp\ipykernel_4064\2669518805.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [20]:
# ── FIG 6: DEV vs EVAL COMPARISON ───────────────────────────────────────────
dev_eer  = ckpt.get('eer', 0.018)
eval_eer = metrics['eer']
dev_acc  = ckpt.get('accuracy', 97.56)
eval_acc = metrics['accuracy']

fig, axes = plt.subplots(1, 2, figsize=(10, 5))

# EER comparison
axes[0].bar(['Dev Set', 'Eval Set'], [dev_eer*100, eval_eer*100],
            color=['#22c55e', '#ef4444'], edgecolor='black', linewidth=0.5)
axes[0].set_ylabel('EER (%)', fontsize=12)
axes[0].set_title('EER: Dev vs Eval', fontsize=13)
axes[0].grid(True, alpha=0.3, axis='y')
for i, (label, val) in enumerate([('Dev', dev_eer*100), ('Eval', eval_eer*100)]):
    axes[0].text(i, val + 0.3, f'{val:.2f}%', ha='center', fontsize=12, fontweight='bold')

# Accuracy comparison
axes[1].bar(['Dev Set', 'Eval Set'], [dev_acc, eval_acc],
            color=['#22c55e', '#f59e0b'], edgecolor='black', linewidth=0.5)
axes[1].set_ylabel('Accuracy (%)', fontsize=12)
axes[1].set_title('Accuracy: Dev vs Eval', fontsize=13)
axes[1].set_ylim(0, 110)
axes[1].grid(True, alpha=0.3, axis='y')
for i, (label, val) in enumerate([('Dev', dev_acc), ('Eval', eval_acc)]):
    axes[1].text(i, val + 1, f'{val:.2f}%', ha='center', fontsize=12, fontweight='bold')

plt.suptitle('Transformer — Dev vs Eval Generalization Gap', fontsize=13, fontweight='bold')
plt.tight_layout()
save_path = FIG_DIR / '06_dev_vs_eval.png'
plt.savefig(save_path, dpi=150, bbox_inches='tight')
plt.show()
print(f'Saved: {save_path}')

Saved: experiments\evaluation_transformer\figures\06_dev_vs_eval.png


C:\Users\PC\AppData\Local\Temp\ipykernel_4064\3566882455.py:32: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


In [21]:
# ── FINAL SUMMARY ────────────────────────────────────────────────────────────
print('\n' + '='*55)
print('ALL FIGURES SAVED')
print('='*55)
for f in sorted(FIG_DIR.glob('*.png')):
    print(f'  {f.name}')

print(f'\nLocation: {FIG_DIR.resolve()}')
print(f'\n{"="*55}')
print('FINAL RESULTS SUMMARY')
print(f'{"="*55}')
print(f'  Model        : Transformer (trained on 100% data)')
print(f'  Eval samples : {metrics["total"]:,}')
print(f'  Dev EER      : {dev_eer*100:.2f}%')
print(f'  Eval EER     : {eval_eer*100:.2f}%')
print(f'  Dev Acc      : {dev_acc:.2f}%')
print(f'  Eval Acc     : {eval_acc:.2f}%')
print(f'  Precision    : {metrics["precision"]:.4f}')
print(f'  Recall       : {metrics["recall"]:.4f}')
print(f'  F1           : {metrics["f1"]:.4f}')
print(f'  TP={metrics["tp"]:,}  TN={metrics["tn"]:,}  FP={metrics["fp"]:,}  FN={metrics["fn"]:,}')


ALL FIGURES SAVED
  01_confusion_matrix.png
  02_score_distribution.png
  03_det_curve.png
  04_per_attack_analysis.png
  05_metrics_summary.png
  06_dev_vs_eval.png

Location: C:\Users\PC\Desktop\Projetcs\ISOC\M1\DeepLearning\projects\voice-manipulation-detection\notebooks\experiments\evaluation_transformer\figures

FINAL RESULTS SUMMARY
  Model        : Transformer (trained on 100% data)
  Eval samples : 71,237
  Dev EER      : 1.80%
  Eval EER     : 26.76%
  Dev Acc      : 97.40%
  Eval Acc     : 87.11%
  Precision    : 0.9566
  Recall       : 0.8970
  F1           : 0.9258
  TP=57,300  TN=4,754  FP=2,601  FN=6,582
